In [1]:
import pandas as pd

pad_varianten = '../../data/H8/word-embeddings/geluidsoverlast_fasttextvarianten.xlsx'
pad_llm = '../../data/H8/llm/geluidsoverlast_llm.xlsx'
pad_definitief = '../../data/H8/geluidsoverlast_w3_nacht_allepersonen.xlsx'

df_var = pd.read_excel(pad_varianten)
df_llm = pd.read_excel(pad_llm)
df_def = pd.read_excel(pad_definitief)

print("Varianten:", df_var.shape)
print("LLM:", df_llm.shape)
print("Definitief:", df_def.shape)

Varianten: (8459, 44)
LLM: (4941, 53)
Definitief: (1823, 72)


In [2]:
nachtcategorieen = [
    'nacht',
    'avondschemering',
    'ochtendschemering'
]

def maak_event_id(df):
    archive = df['archive_file'].astype(str).str.strip()
    incident = (
        pd.to_numeric(df['Incident Number'], errors='coerce')
        .astype('Int64')
        .astype(str)
    )
    return archive + '_' + incident

In [3]:
var_nacht = df_var[
    (df_var['district'] == 3) &
    (df_var['astro'].isin(nachtcategorieen))
].copy()

var_nacht['event_id'] = maak_event_id(var_nacht)

var_events = set(var_nacht['event_id'])

print("Gebeurtenissen variantenlijst:", len(var_events))

Gebeurtenissen variantenlijst: 742


In [4]:
labels = [
    'disrupting nightrest',
    'doorbell ringing',
    'making noise',
    'other_sound',
    'playing inn music',
    'playing street music',
    'scolding',
    'singing',
    'traffic noise',
    'yelling'
]

llm_nacht = df_llm[
    df_llm['astronomical_category'].isin(nachtcategorieen)
].copy()

llm_nacht['event_id'] = maak_event_id(llm_nacht)

llm_event = (
    llm_nacht
    .groupby('event_id')[labels]
    .max()
    .reset_index()
)

In [5]:
llm_overlast = llm_event[
    llm_event['making noise'] |
    llm_event['disrupting nightrest']
].copy()

llm_events = set(llm_overlast['event_id'])

print("LLM-gebeurtenissen:", len(llm_events))

LLM-gebeurtenissen: 740


In [6]:
overlap = llm_events & var_events
alleen_llm = llm_events - var_events
alleen_varianten = var_events - llm_events

print("LLM:", len(llm_events))
print("Varianten:", len(var_events))
print("Overlap:", len(overlap))
print("Alleen LLM:", len(alleen_llm))
print("Alleen varianten:", len(alleen_varianten))

print(
    "\nLLM ook via varianten:",
    f"{len(overlap) / len(llm_events) * 100:.1f}%"
)

print(
    "Varianten ook via LLM:",
    f"{len(overlap) / len(var_events) * 100:.1f}%"
)

LLM: 740
Varianten: 742
Overlap: 594
Alleen LLM: 146
Alleen varianten: 148

LLM ook via varianten: 80.3%
Varianten ook via LLM: 80.1%


In [7]:
df_def['event_id'] = maak_event_id(df_def)

df_def_event = (
    df_def
    .drop_duplicates('event_id')
    .copy()
)

def_events = set(df_def_event['event_id'])

print("Definitieve dataset:", len(def_events))

Definitieve dataset: 819


In [8]:
var_true = var_events & def_events
var_false = var_events - def_events

print("Varianten totaal:", len(var_events))
print("Werkelijk geluidsoverlast:", len(var_true))
print("Valse positieven:", len(var_false))

print(
    "Precisie:",
    f"{len(var_true) / len(var_events) * 100:.1f}%"
)

print(
    "Dekking definitieve dataset:",
    f"{len(var_true) / len(def_events) * 100:.1f}%"
)

Varianten totaal: 742
Werkelijk geluidsoverlast: 672
Valse positieven: 70
Precisie: 90.6%
Dekking definitieve dataset: 82.1%


In [9]:
alleen_var_true = alleen_varianten & def_events
alleen_var_false = alleen_varianten - def_events

print("Alleen varianten:", len(alleen_varianten))
print("Uiteindelijk behouden:", len(alleen_var_true))
print("Uiteindelijk niet behouden:", len(alleen_var_false))

Alleen varianten: 148
Uiteindelijk behouden: 80
Uiteindelijk niet behouden: 68


In [10]:
andere_labels = [
    'doorbell ringing',
    'other_sound',
    'playing inn music',
    'playing street music',
    'scolding',
    'singing',
    'traffic noise',
    'yelling'
]

behouden_var = llm_event[
    llm_event['event_id'].isin(alleen_var_true)
].copy()

ids_ander_llm = set(
    behouden_var.loc[
        behouden_var[andere_labels].any(axis=1),
        'event_id'
    ]
)

ids_handmatig = alleen_var_true - ids_ander_llm

print("Behouden alleen-varianten:", len(alleen_var_true))
print("Met ander LLM-geluidslabel:", len(ids_ander_llm))
print("Via varianten + handmatige controle:", len(ids_handmatig))

Behouden alleen-varianten: 80
Met ander LLM-geluidslabel: 58
Via varianten + handmatige controle: 22


In [11]:
print("LLM algemeen:", len(llm_events))
print("Extra via ander LLM-label:", len(ids_ander_llm))
print("Extra via handmatige controle:", len(ids_handmatig))

totaal = (
    len(llm_events) +
    len(ids_ander_llm) +
    len(ids_handmatig)
)

print("\nTotaal:", totaal)
print("Definitieve dataset:", len(def_events))

LLM algemeen: 740
Extra via ander LLM-label: 58
Extra via handmatige controle: 22

Totaal: 820
Definitieve dataset: 819


In [12]:
# ---------------------------------------------------------
# 1. Definitieve dataset wijk 3
# ---------------------------------------------------------

df_def['event_id'] = maak_event_id(df_def)
# lege rijen (zonder archiefnummer) tellen niet mee
df_def_geldig = df_def[df_def['archive_file'].notna()]

print("Totaal definitieve dataset:", df_def_geldig['event_id'].nunique())


# ---------------------------------------------------------
# 2. Specifieke geluidspraktijken
# ---------------------------------------------------------

praktijklabels = [
    'yelling',
    'singing',
    'arguing',
    'playing inn music',
    'scolding',
    'playing street music',
    'doorbell ringing',
    'traffic noise'
]

# per gebeurtenis: een praktijk telt zodra minstens één betrokken persoon dat label heeft
heeft_praktijklabel = (
    df_def_geldig[praktijklabels]
    .fillna(False)
    .astype(bool)
    .groupby(df_def_geldig['event_id'])
    .any()
    .any(axis=1)
)

n_praktijk = heeft_praktijklabel.sum()
n_totaal = len(heeft_praktijklabel)

print(
    "Met specifiek praktijklabel:",
    n_praktijk,
    f"({n_praktijk / n_totaal * 100:.1f}%)"
)


# ---------------------------------------------------------
# 3. Macroset variantenlijst
# ---------------------------------------------------------

df_var['event_id'] = maak_event_id(df_var)

macro_totaal = (
    df_var[
        df_var['astro'].notna()
    ]['event_id']
    .nunique()
)

macro_nacht = (
    df_var[
        df_var['astro'].isin(nachtcategorieen)
    ]['event_id']
    .nunique()
)

print("Macro totaal:", macro_totaal)
print("Macro nacht:", macro_nacht)

Totaal definitieve dataset: 818
Met specifiek praktijklabel: 492 (60.1%)
Macro totaal: 2221
Macro nacht: 1722


In [13]:
niet_in_def = (llm_events & var_events) - def_events
niet_in_bronnen = def_events - llm_events - var_events

print("Overlap niet in definitief:", sorted(niet_in_def))
print("Definitief zonder bron:", sorted(niet_in_bronnen))
print("ID's met <NA>:", sum('<NA>' in e for e in def_events | llm_events | var_events))

Overlap niet in definitief: ['731#1595_755', 'MA#17618_4']
Definitief zonder bron: ['nan_<NA>']
ID's met <NA>: 1


In [14]:
df_def[df_def['archive_file'].isna() | df_def['Incident Number'].isna()]

,archive_file,year,district,Person ID,Incident Number,Page Number,E_DATE,E_TIME,E_LOC,P_FIRSTNAME,...,scolding,silence,singing,traffic noise,whistling,yelling,geluidsoverlast_y,geluidsoverlast,bronbestand,event_id
1822,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan_<NA>


In [15]:
ids = ['731#1595_755', 'MA#17618_4', 'MA#17633_665']
for bron, df in [('llm', llm_nacht), ('var', var_nacht), ('def', df_def)]:
    rij = df[df['event_id'].isin(ids)]
    print(bron, rij[['event_id', 'Incident Number']].drop_duplicates().values.tolist())

llm [['731#1595_755', 755], ['MA#17618_4', 4]]
var [['731#1595_755', 755], ['MA#17618_4', 4]]
def []


In [16]:
llm_nacht.loc[llm_nacht['event_id'].isin(['731#1595_755', 'MA#17618_4']),
              ['event_id', 'district', 'astronomical_category', 'Person ID']]

,event_id,district,astronomical_category,Person ID
276,731#1595_755,3,nacht,1021
3529,MA#17618_4,3,nacht,19
3530,MA#17618_4,3,nacht,20
